In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorForTokenClassification,
)
import numpy as np
import evaluate
import matplotlib.pyplot as plt

In [ ]:
dataset = load_dataset("eriktks/conll2003", revision="convert/parquet")
print(dataset)

In [ ]:
model_id = "model"
ner_model_id = "ner"

tokenizer = AutoTokenizer.from_pretrained(model_id)

label_names = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_names)

model = AutoModelForTokenClassification.from_pretrained(
    model_id,
    num_labels=num_labels,
    ignore_mismatched_sizes=True,
)

model_size = sum(t.numel() for t in model.parameters())
print(f"Transformer size: {model_size/1000**2:.1f}M parameters")

model.config.id2label = {i: name for i, name in enumerate(label_names)}
model.config.label2id = {name: i for i, name in enumerate(label_names)}

print(label_names)

In [ ]:
def tokenize(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
    )

    all_labels = []

    for i, labels in enumerate(example["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized["labels"] = all_labels
    return tokenized

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns([
    "id", "tokens", "pos_tags", "chunk_tags", "ner_tags"
])
dataset.set_format("torch")

print(dataset)

In [ ]:
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for pred, label in zip(preds, labels):
        current_preds = []
        current_labels = []

        for p, l in zip(pred, label):
            if l != -100:
                current_preds.append(label_names[p])
                current_labels.append(label_names[l])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

training_args = TrainingArguments(
    output_dir=ner_model_id,
    learning_rate=2e-5,
    weight_decay=0.02,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=5,

    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",

    logging_steps=200,
    eval_steps=200,
    save_steps=200,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    report_to="none",
    fp16=True,
    seed=42,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(ner_model_id)

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)


In [ ]:
logs = trainer.state.log_history

train_steps = [log["step"] for log in logs if "loss" in log and "eval_loss" not in log]
train_losses = [log["loss"] for log in logs if "loss" in log and "eval_loss" not in log]

eval_steps = [log["step"] for log in logs if "eval_loss" in log]
eval_losses = [log["eval_loss"] for log in logs if "eval_loss" in log]

plt.figure(figsize=(12, 6))
plt.plot(train_steps, train_losses, label="Training Loss", linewidth=2)
plt.plot(eval_steps, eval_losses, label="Validation Loss", linewidth=2)
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()


In [ ]:
test_metrics = trainer.evaluate(dataset["test"])
print(test_metrics)

In [ ]:
from transformers import pipeline

ner = pipeline(
    "token-classification",
    model=ner_model_id,
    tokenizer=ner_model_id,
    aggregation_strategy="simple",
)

text = "Barack Obama visited Paris and met Microsoft executives."
print(ner(text))